In [7]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/processed/pit_within_2_laps.parquet")
print(df.shape)
print(df["PitWithinNextNLaps"].value_counts())


(2586, 15)
PitWithinNextNLaps
0    2572
1      14
Name: count, dtype: int64


In [8]:
y = df["PitWithinNextNLaps"].astype(int)

X = df.drop(columns=[
    "PitWithinNextNLaps",
    "Driver",
    "Season",
    "GP"
])

# ensure bool → int
for col in X.columns:
    if X[col].dtype == "bool":
        X[col] = X[col].astype(int)

print(X.dtypes)


LapNumber          float64
LapTimeSec         float64
S1Sec              float64
S2Sec              float64
S3Sec              float64
TyreLife           float64
LapDeltaPrev       float64
LapMean3           float64
LapStd3            float64
Compound_MEDIUM      int64
Compound_SOFT        int64
dtype: object


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train label balance:")
print(y_train.value_counts())
print("\nTest label balance:")
print(y_test.value_counts())


Train label balance:
PitWithinNextNLaps
0    2057
1      11
Name: count, dtype: int64

Test label balance:
PitWithinNextNLaps
0    515
1      3
Name: count, dtype: int64


In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained.")


Model trained.


In [11]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred = model.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Baseline F1:", round(f1_score(y_test, y_pred), 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))


Baseline Accuracy: 0.994
Baseline F1: 0.0

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       515
           1       0.00      0.00      0.00         3

    accuracy                           0.99       518
   macro avg       0.50      0.50      0.50       518
weighted avg       0.99      0.99      0.99       518



In [12]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score
)

y_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_proba), 3))

prec, rec, thr = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-9)

best_idx = np.argmax(f1_scores)
best_thr = thr[max(best_idx - 1, 0)]

print("Best threshold:", round(best_thr, 4))
print("Best F1:", round(f1_scores[best_idx], 3))


ROC-AUC: 0.617
PR-AUC: 0.028
Best threshold: 0.0533
Best F1: 0.118


In [13]:
y_pred_tuned = (y_proba >= best_thr).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_tuned, zero_division=0))


              precision    recall  f1-score   support

           0       1.00      0.97      0.98       515
           1       0.07      0.33      0.11         3

    accuracy                           0.97       518
   macro avg       0.53      0.65      0.55       518
weighted avg       0.99      0.97      0.98       518

